In [1]:
# =====================================================
# Cell 1: System Info & Resource Versions
# =====================================================
# This cell shows basic system info—OS version, GPU, default Python,
# pip, conda (if installed) etc. It's useful for debugging or for
# reproducibility info in a shared notebook.

!set -euxo pipefail

print("=== OS Release Info ===")
!lsb_release -a

print("\n=== GPU Info (nvidia-smi) ===")
!nvidia-smi || echo "[WARNING] nvidia-smi not found"

print("\n=== Default nvcc Version (if available) ===")
!nvcc --version || echo "[WARNING] nvcc not found"

print("\n=== Default Python & Pip Version ===")
!which python
!python --version
!pip --version

print("\n=== Conda Version (if any) ===")
!conda --version || echo "[INFO] 'conda' not yet installed."


=== OS Release Info ===
No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 24.04.4 LTS
Release:	24.04
Codename:	noble

=== GPU Info (nvidia-smi) ===
Fri Oct 30 20:23:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  153

In [5]:
# =====================================================
# Cell 2: Install CUDA 11.8 Toolkit
# =====================================================

import os
import subprocess

CUDA_ROOT = "/usr/local/cuda-11.8"
NVCC = f"{CUDA_ROOT}/bin/nvcc"
RUNFILE = "/content/cuda_11.8.0_520.61.05_linux.run"
RUNFILE_URL = (
    "https://developer.download.nvidia.com/compute/cuda/11.8.0/"
    "local_installers/cuda_11.8.0_520.61.05_linux.run"
)

print("---- Checking for CUDA 11.8 compiler ----")

cuda_118_ready = False
if os.path.isfile(NVCC):
    result = subprocess.run(
        [NVCC, "--version"],
        check=True,
        capture_output=True,
        text=True
    )
    print(result.stdout)
    cuda_118_ready = "release 11.8" in result.stdout

if not cuda_118_ready:
    print("---- Installing CUDA 11.8 toolkit ----")

    # STEP 1) Install required tools
    apt_env = os.environ.copy()
    apt_env["LD_LIBRARY_PATH"] = (
        "/usr/lib/x86_64-linux-gnu:/usr/lib64-nvidia"
    )

    subprocess.run(["apt-get", "update", "-y"], env=apt_env, check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "build-essential", "wget"],
        env=apt_env,
        check=True
    )

    # STEP 2) Download the official CUDA 11.8 runfile
    subprocess.run(
        ["wget", "-q", "-c", "-O", RUNFILE, RUNFILE_URL],
        check=True
    )

    # STEP 3) Install the toolkit only; keep Colab's existing driver
    subprocess.run(
        ["sudo", "sh", RUNFILE, "--silent", "--toolkit", "--override"],
        check=True
    )

    # STEP 4) Remove the installer
    if os.path.exists(RUNFILE):
        os.remove(RUNFILE)

# STEP 5) Set CUDA 11.8 paths for later cells
os.environ["CUDA_HOME"] = CUDA_ROOT
os.environ["PATH"] = f"{CUDA_ROOT}/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = (
    f"{CUDA_ROOT}/lib64:/usr/lib64-nvidia:/usr/lib/x86_64-linux-gnu"
)

# STEP 6) Verify the compiler version
result = subprocess.run(
    [NVCC, "--version"],
    check=True,
    capture_output=True,
    text=True
)
print(result.stdout)

if "release 11.8" not in result.stdout:
    raise RuntimeError("CUDA 11.8 was not installed correctly.")

print("[SUCCESS] CUDA 11.8 is ready.")

---- Checking for CUDA 11.8 compiler ----
---- Installing CUDA 11.8 toolkit ----
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2022 NVIDIA Corporation
Built on Wed_Sep_21_10:33:58_PDT_2022
Cuda compilation tools, release 11.8, V11.8.89
Build cuda_11.8.r11.8/compiler.31833905_0

[SUCCESS] CUDA 11.8 is ready.


In [3]:
# =====================================================
# Cell 3: Double-Check CUDA 11.8 (Optional)
# =====================================================
# In Colab, environment variables may not persist across cells.
# This cell verifies that 'nvcc' is still available and at 11.8.

!export PATH="/usr/local/cuda-11.8/bin:$PATH" && \
nvcc --version


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [4]:
# =====================================================
# Cell 4: Install Miniconda using condacolab
# =====================================================
# We use `condacolab` to install and configure a Miniconda environment
# inside Colab. Once installed, we still need to "source" and activate
# it in bash cells to use conda commands.

!pip install -q condacolab
import condacolab
condacolab.install_miniconda()



📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

⏬ Downloading https://repo.anaconda.com/miniconda/Miniconda3-py313_26.5.3-2-Linux-x86_64.sh...


Miniconda is subject to terms of service: https://anaconda.com/legal/terms/terms-of-service


📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:31
🔁 Restarting kernel...


In [2]:
%%bash
set -euo pipefail

# =====================================================
# Cell 5: Create & Populate the 'trellis' Environment
# =====================================================

# STEP 1) Activate Conda and set CUDA 11.8 paths
source /usr/local/etc/profile.d/conda.sh

export CUDA_HOME="/usr/local/cuda-11.8"
export PATH="$CUDA_HOME/bin:$PATH"
export LD_LIBRARY_PATH="$CUDA_HOME/lib64:/usr/lib64-nvidia:/usr/lib/x86_64-linux-gnu"

# Keep package versions compatible with this TRELLIS setup
cat > /tmp/trellis-constraints.txt <<'EOF'
numpy<2
Pillow<11
opencv-python-headless<5
rembg<2.0.68
EOF
export PIP_CONSTRAINT="/tmp/trellis-constraints.txt"

# STEP 2) Create the Python 3.11 environment using conda-forge only
if ! conda env list | awk '$1 == "trellis" { found=1 } END { exit !found }'; then
    conda create -y -n trellis \
        --override-channels -c conda-forge \
        python=3.11 pip
else
    echo "[INFO] Environment 'trellis' already exists; reusing it."
fi

conda activate trellis

# STEP 3) Install PyTorch 2.3.0 with CUDA 11.8 support
python -m pip install --upgrade pip setuptools wheel
python -m pip install \
    --index-url https://download.pytorch.org/whl/cu118 \
    torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0

# STEP 4) Install TRELLIS-compatible extensions and UI packages
python -m pip install \
    https://github.com/Dao-AILab/flash-attention/releases/download/v2.5.8/flash_attn-2.5.8+cu118torch2.3cxx11abiFALSE-cp311-cp311-linux_x86_64.whl

python -m pip install spconv-cu118

python -m pip install \
    kaolin==0.17.0 \
    -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.3.0_cu118.html

python -m pip install "Pillow<11"
python -m pip install gradio==4.44.1 gradio_litmodel3d==0.0.1

echo "[INFO] 'trellis' environment created and packages installed."
python --version
python -c "import torch; print('PyTorch:', torch.__version__); print('CUDA available:', torch.cuda.is_available())"

Jupyter detected...
Retrieving notices: - done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: \ | done

## Package Plan ##

  environment location: /usr/local/envs/trellis

  added / updated specs:
    - pip
    - python=3.11


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-1.0.8                |      hda65f42_10         252 KB  conda-forge
    ca-certificates-2026.10.23 |       hbd8a1cb_0         127 KB  conda-forge
    ld_impl_linux-64-2.46.1    |default_hbd61a6d_103         725 KB  conda-forge
    libexpat-2.9.0             |       hd2095e1_0          77 KB  conda-forge
    libffi-3.7.0               |       h81df57d_1          66 KB  conda-forge
    libgcc-16.2.0              |       ha9f2e26_7         1.0 MB  conda-forge
    libgomp-16.2.0             |       he0feb66_7      

bash: /usr/local/lib/libtinfo.so.6: no version information available (required by bash)
WARNING conda.conda_pypi.main:notify_externally_managed_future(156): 
  Did you know? You can install many PyPI packages with conda
  using the conda-pypi beta. Get started:
    https://docs.conda.io/projects/conda/en/stable/new-features.html



In [6]:
%%bash
set -euo pipefail

# =====================================================
# Cell 6: System Dev Tools & Python Build Libraries
# =====================================================

# STEP 1) Install system build tools, without another CUDA toolkit
env LD_LIBRARY_PATH="/usr/lib/x86_64-linux-gnu:/usr/lib64-nvidia" apt-get update -y
env LD_LIBRARY_PATH="/usr/lib/x86_64-linux-gnu:/usr/lib64-nvidia" \
    apt-get install -y build-essential cmake ninja-build

# STEP 2) Activate the 'trellis' environment
source /usr/local/etc/profile.d/conda.sh
conda activate trellis

# STEP 3) Select CUDA 11.8 for builds
export CUDA_HOME="/usr/local/cuda-11.8"
export PATH="$CUDA_HOME/bin:$PATH"
export LD_LIBRARY_PATH="$CUDA_HOME/lib64:/usr/lib64-nvidia:/usr/lib/x86_64-linux-gnu"

if [ ! -x "$CUDA_HOME/bin/nvcc" ]; then
    echo "[ERROR] nvcc was not found at $CUDA_HOME/bin/nvcc"
    exit 1
fi

# STEP 4) Upgrade Python build libraries
python -m pip install --upgrade pip setuptools wheel ninja Cython

# STEP 5) Confirm the compiler version
"$CUDA_HOME/bin/nvcc" --version
echo "[INFO] System dev tools and Python build libraries installed."

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:5 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:9 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
build-essential is already the newest version (12.10ubuntu1).
cmake is already the newest version (3.28.3-1bui

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


May crash at this point. if it does, just resume in the cell below


In [8]:
%%bash
set -euo pipefail

# =====================================================
# Cell 7: Clone & Setup Microsoft/TRELLIS
# =====================================================

# STEP 1) Activate 'trellis' and select CUDA 11.8
source /usr/local/etc/profile.d/conda.sh
conda activate trellis

export CUDA_HOME="/usr/local/cuda-11.8"
export PATH="$CUDA_HOME/bin:$PATH"
export LD_LIBRARY_PATH="$CUDA_HOME/lib64:/usr/lib64-nvidia:/usr/lib/x86_64-linux-gnu"

if [ ! -x "$CUDA_HOME/bin/nvcc" ]; then
    echo "[ERROR] CUDA 11.8 nvcc not found at $CUDA_HOME/bin/nvcc"
    exit 1
fi

# STEP 2) Install and select GCC 11 for CUDA extension builds
env LD_LIBRARY_PATH="/usr/lib/x86_64-linux-gnu:/usr/lib64-nvidia" \
    apt-get install -y gcc-11 g++-11

export CC="/usr/bin/gcc-11"
export CXX="/usr/bin/g++-11"
export CUDAHOSTCXX="/usr/bin/g++-11"
export NVCC_PREPEND_FLAGS="-ccbin=/usr/bin/g++-11"
export CMAKE_CUDA_HOST_COMPILER="/usr/bin/g++-11"

# STEP 3) Keep package versions compatible with this environment
cat > /tmp/trellis-constraints.txt <<'EOF'
numpy<2
opencv-python-headless<4.12
Pillow<11
rembg<2.0.68
EOF
export PIP_CONSTRAINT="/tmp/trellis-constraints.txt"

# STEP 4) Clone Microsoft/TRELLIS with its submodules
if [ ! -d /content/TRELLIS/.git ]; then
    if [ -e /content/TRELLIS ]; then
        rm -rf /content/TRELLIS
    fi
    git clone --recurse-submodules \
        https://github.com/microsoft/TRELLIS.git \
        /content/TRELLIS
else
    echo "[INFO] TRELLIS repository already exists; updating submodules."
    git -C /content/TRELLIS submodule update --init --recursive
fi

cd /content/TRELLIS
mkdir -p assets/example_image

# STEP 5) Install basic and demo dependencies
bash ./setup.sh --basic --demo

# STEP 6) Install nvdiffrast without build isolation
mkdir -p /tmp/extensions

if [ ! -d /tmp/extensions/nvdiffrast/.git ]; then
    if [ -e /tmp/extensions/nvdiffrast ]; then
        rm -rf /tmp/extensions/nvdiffrast
    fi
    git clone https://github.com/NVlabs/nvdiffrast.git \
        /tmp/extensions/nvdiffrast
fi

python -m pip install --no-build-isolation -v \
    /tmp/extensions/nvdiffrast

# STEP 7) Confirm the compiler versions
/usr/bin/gcc-11 --version
"$CUDA_HOME/bin/nvcc" --version

echo "[INFO] TRELLIS setup completed."

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  cpp-11 gcc-11-base libasan6 libgcc-11-dev libstdc++-11-dev libtsan0
Suggested packages:
  gcc-11-locales g++-11-multilib gcc-11-doc gcc-11-multilib libstdc++-11-doc
The following NEW packages will be installed:
  cpp-11 g++-11 gcc-11 gcc-11-base libasan6 libgcc-11-dev libstdc++-11-dev
  libtsan0
0 upgraded, 8 newly installed, 0 to remove and 54 not upgraded.
Need to get 50.7 MB of archives.
After this operation, 160 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 gcc-11-base amd64 11.5.0-1ubuntu1~24.04.1 [45.6 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 cpp-11 amd64 11.5.0-1ubuntu1~24.04.1 [9,999 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 libasan6 amd64 11.5.0-1ubuntu1~24.04.1 [2,264 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble-up

  Running command git clone --filter=blob:none --quiet https://github.com/EasternJournalist/utils3d.git /tmp/pip-req-build-4by44938
  Running command git rev-parse -q --verify '9a4eb15e4021b67b12c460c7057d642626897ec8^{commit}'
  9a4eb15e4021b67b12c460c7057d642626897ec8
  Running command git checkout -q 9a4eb15e4021b67b12c460c7057d642626897ec8
  Running command Preparing metadata (pyproject.toml)
  running dist_info
  creating /tmp/pip-modern-metadata-mj9vfblq/nvdiffrast.egg-info
  writing /tmp/pip-modern-metadata-mj9vfblq/nvdiffrast.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-modern-metadata-mj9vfblq/nvdiffrast.egg-info/dependency_links.txt
  writing requirements to /tmp/pip-modern-metadata-mj9vfblq/nvdiffrast.egg-info/requires.txt
  writing top-level names to /tmp/pip-modern-metadata-mj9vfblq/nvdiffrast.egg-info/top_level.txt
  writing manifest file '/tmp/pip-modern-metadata-mj9vfblq/nvdiffrast.egg-info/SOURCES.txt'
  reading manifest file '/tmp/pip-modern-metadata-mj9vf

You may see warnings in the out of cell below. safe to ignore.

In [9]:
%%bash
set -euo pipefail

# =====================================================
# Cell 8: Clone & Install mip-splatting
# =====================================================

# STEP 1) Clone mip-splatting with its submodules
cd /content

if [ ! -d "/content/mip-splatting/.git" ]; then
    if [ -e "/content/mip-splatting" ]; then
        rm -rf /content/mip-splatting
    fi

    git clone --recurse-submodules \
        https://github.com/autonomousvision/mip-splatting.git \
        /content/mip-splatting
else
    echo "[INFO] 'mip-splatting' already exists; updating submodules."
    git -C /content/mip-splatting submodule update --init --recursive
fi

# STEP 2) Install required system packages
env LD_LIBRARY_PATH="/usr/lib/x86_64-linux-gnu:/usr/lib64-nvidia" \
    apt-get update -y

env LD_LIBRARY_PATH="/usr/lib/x86_64-linux-gnu:/usr/lib64-nvidia" \
    apt-get install -y libgl1-mesa-dev build-essential cmake ninja-build

# STEP 3) Activate 'trellis' and select CUDA 11.8
source /usr/local/etc/profile.d/conda.sh
conda activate trellis

export CUDA_HOME="/usr/local/cuda-11.8"
export PATH="$CUDA_HOME/bin:$PATH"
export LD_LIBRARY_PATH="$CUDA_HOME/lib64:/usr/lib64-nvidia:/usr/lib/x86_64-linux-gnu"

# STEP 4) Select GCC 11 for CUDA extension builds
export CC="/usr/bin/gcc-11"
export CXX="/usr/bin/g++-11"
export CUDAHOSTCXX="/usr/bin/g++-11"
export NVCC_PREPEND_FLAGS="-ccbin=/usr/bin/g++-11"
export CMAKE_CUDA_HOST_COMPILER="/usr/bin/g++-11"

export PIP_CONSTRAINT="/tmp/trellis-constraints.txt"

# STEP 5) Build diff_gaussian_rasterization
cd /content/mip-splatting/submodules/diff-gaussian-rasterization

python -m pip install . --no-build-isolation -v

echo "[INFO] diff_gaussian_rasterization installed successfully."

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
build-essential is already the newest version (12.10ubuntu1).
cmake is already the newest version (3.28.3-1bui

Cloning into '/content/mip-splatting'...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
  Running command Preparing metadata (pyproject.toml)
  running dist_info
  creating /tmp/pip-modern-metadata-vmjam421/diff_gaussian_rasterization.egg-info
  writing /tmp/pip-modern-metadata-vmjam421/diff_gaussian_rasterization.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-modern-metadata-vmjam421/diff_gaussian_rasterization.egg-info/dependency_links.txt
  writing top-level names to /tmp/pip-modern-metadata-vmjam421/diff_gaussian_rasterization.egg-info/top_level.txt
  writing manifest file '/tmp/pip-modern-metadata-vmjam421/diff_gaussian_rasterization.egg-info/SOURCES.txt'
  reading manifest file '/tmp/pip-modern-metadata-vmjam421/diff_gaussian_rasterization.egg-info/SOURCES.txt'
  adding license file 'LICENSE.md'
  writing manifest file '/tmp/pip-m

**RUN TRELLIS IN NGROK (PROVIDE YOUR NGROK API KEY)**\
if you run the code below, dont forget to edit and put in your nGrok Auth token\
Also, by default, we load kaiser-app.py \
if you want original app.py, uncomment that line and comment out kaiser-app.py


In [ ]:
# =====================================================
# Cell 9: Launch TRELLIS App with Ngrok (show URL after launch)
# =====================================================
# 0) Download kaiser-app.py from your GitHub (if not present)
# 1) Install/Update ngrok if not present
# 2) Configure ngrok with your token (handle new vs. old versions)
# 3) (Re)install Gradio in 'trellis'
# 4) Start ngrok to expose port 7860
# 5) Launch either 'app.py' or 'kaiser-app.py' in 'trellis' environment
# 6) Print ngrok URL AFTER the Gradio app is actually running

import subprocess, time, json, re, os

# ============== STEP 0) Download kaiser-app.py ==============
# We'll place kaiser-app.py in /content/TRELLIS if it's not already there.
TARGET_DIR = "/content/TRELLIS"
kaiser_app_path = os.path.join(TARGET_DIR, "kaiser-app.py")


print("[INFO] Downloading kaiser-app.py from GitHub into /content/TRELLIS ...")
# Note: We use the raw GitHub link to fetch the raw file contents.
# If you change branches or repos, update this URL accordingly.
KAISER_APP_URL = "https://raw.githubusercontent.com/jackel27/Trellis-Colab/main/kaiser-app.py"
subprocess.run([
    "wget", "-q", "-O", kaiser_app_path, KAISER_APP_URL
], check=True)


# ============== USER CHOICE ==============
# Uncomment exactly ONE of the following lines
# to select which app file you want to run.

#APP_FILE = "app.py" # <-- This is TRELLIS Default app.py
APP_FILE = "kaiser-app.py"  # from https://github.com/jackel27/Trellis-Colab/blob/main/kaiser-app.py

# =========================================


#---------------NGROK AUTH TOKEN-----------
# Replace with your own ngrok auth token
NGROK_AUTH = "PLACE NGROK AUTH TOKEN HERE"
#---------------NGROK AUTH TOKEN-----------


# Step 1) Check if ngrok is installed; if not, download & install
try:
    subprocess.run(["which", "ngrok"], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print("[INFO] ngrok is already installed.")
except subprocess.CalledProcessError:
    print("[INFO] Installing ngrok ...")
    subprocess.run(["wget", "-q", "https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip"], check=True)
    subprocess.run(["unzip", "-o", "ngrok-stable-linux-amd64.zip"], check=True)
    subprocess.run(["chmod", "+x", "ngrok"], check=True)
    subprocess.run(["mv", "ngrok", "/usr/local/bin/ngrok"], check=True)

# (Optional) Print ngrok version to see which command we might need
try:
    print("[INFO] Checking ngrok version...")
    subprocess.run(["ngrok", "version"], check=True)
except:
    print("[WARNING] Could not determine ngrok version.")

# Step 2) Configure ngrok (new vs. old command)
try:
    print("[INFO] Configuring ngrok with new command: 'ngrok config add-authtoken' ...")
    subprocess.run(["ngrok", "config", "add-authtoken", NGROK_AUTH], check=True)
except subprocess.CalledProcessError:
    print("[WARNING] 'ngrok config add-authtoken' failed. Trying older command 'ngrok authtoken' ...")
    subprocess.run(["ngrok", "authtoken", NGROK_AUTH], check=True)

# Step 3) (Re)install Gradio in 'trellis'
print("[INFO] Ensuring Gradio is installed in 'trellis' environment...")
subprocess.run([
    "conda", "run", "-n", "trellis",
    "pip", "install", "gradio==4.44.1", "gradio_litmodel3d==0.0.1"
], check=True)

# Step 4) Start ngrok in background, forwarding port 7860
print("[INFO] Starting ngrok on port 7860 ...")
proc_ngrok = subprocess.Popen(["ngrok", "http", "7860"])

# Give ngrok a few seconds to initialize
time.sleep(4)

# Parse ngrok's local API to get the public URL
public_url = None
try:
    tunnels_json = subprocess.check_output(["curl", "-s", "127.0.0.1:4040/api/tunnels"])
    public_url = json.loads(tunnels_json)["tunnels"][0]["public_url"]
except Exception as e:
    print("[ERROR] Could not get ngrok public URL:", e)

# We'll store the URL now, but only print it after the app is ready
if public_url:
    print("[INFO] Found NGROK URL: ", public_url)
    print("[INFO] Ngrok may not be ready yet; you may need to wait until the app is launched.")
else:
    print("[WARNING] No public URL found from ngrok.")

# Step 5) Launch the selected app file in 'trellis' environment, unbuffered
app_path = os.path.join(TARGET_DIR, APP_FILE)  # e.g. /content/TRELLIS/kaiser-app.py
print(f"[INFO] Launching {app_path} in 'trellis' environment ...")

p = subprocess.Popen(
    ["conda", "run", "-n", "trellis", "python", "-u", app_path],
    cwd=TARGET_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True  # so we can read line by line as strings
)

# Step 6) Stream logs. Only print the final NGROK URL after the app is launched
launch_detected = False

try:
    for line in iter(p.stdout.readline, ""):
        if not line:
            break

        # Print the line to the notebook output
        print(line, end="")

        # If we detect "Running on local URL:", we assume the Gradio app is up
        if "Running on local URL:" in line:
            launch_detected = True
            if public_url:
                print(f"\n[INFO] {APP_FILE} is now live!!! Access via ngrok here:\n{public_url}\n")
            else:
                print("\n[WARNING] We have not found an ngrok URL, but the app claims to be running.\n")

except KeyboardInterrupt:
    print("[INFO] Interrupted by user. Stopping the app.")
    p.kill()
finally:
    p.wait()

if not launch_detected and public_url:
    print("[INFO] The app did not print 'Running on local URL:' but here's the NGROK URL anyway:")
    print(public_url)

print("[INFO] Done. If you want to relaunch, re-run this cell.")


**RUN WITH CLOUDFLARE - NO AUTH KEY NEEDED**

In [10]:
# =====================================================
# Cell 10: Launch TRELLIS App with Cloudflare (show URL after launch)
# =====================================================
# 0) Download kaiser-app.py from your GitHub (if not present)
# 1) Install cloudflared (if not already)
# 2) (Re)install Gradio in 'trellis' environment
# 3) Start a Cloudflare "Quick Tunnel" on localhost:7860
# 4) Launch either 'app.py' or 'kaiser-app.py' in 'trellis' environment
# 5) Print the tunnel URL AFTER the app is actually up.

import subprocess
import time
import re
import os

# ============== STEP 0) Download kaiser-app.py ==============
TARGET_DIR = "/content/TRELLIS"
kaiser_app_path = os.path.join(TARGET_DIR, "kaiser-app.py")

print("[INFO] Downloading kaiser-app.py from GitHub into /content/TRELLIS ...")
KAISER_APP_URL = "https://raw.githubusercontent.com/jackel27/Trellis-Colab/main/kaiser-app.py"
subprocess.run(["wget", "-q", "-O", kaiser_app_path, KAISER_APP_URL], check=True)

# ============== USER CHOICE ==============
# Uncomment exactly ONE of the following lines
# to select which app file you want to run.

#APP_FILE = "app.py"
APP_FILE = "kaiser-app.py"
# =========================================

# Step 1) Install/Update cloudflared via the latest .deb for Linux (amd64)
print("[INFO] Installing/Updating cloudflared...")
try:
    # Download the latest .deb from GitHub
    subprocess.run([
        "wget", "-q", "-O", "cloudflared-linux-amd64.deb",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"
    ], check=True)
    # Attempt dpkg install
    subprocess.run(["sudo", "dpkg", "-i", "cloudflared-linux-amd64.deb"], check=True)
except subprocess.CalledProcessError:
    print("[WARNING] cloudflared may already be installed or dpkg encountered an error.")

# Step 2) (Re)install Gradio in 'trellis' environment
print("[INFO] Ensuring Gradio is installed in 'trellis' environment...")
subprocess.run([
    "conda", "run", "-n", "trellis",
    "pip", "install", "gradio==4.44.1", "gradio_litmodel3d==0.0.1"
], check=True)

# Step 3) Start cloudflared in a subprocess to create a Quick Tunnel from :7860
print("[INFO] Starting cloudflared (Quick Tunnel) on port 7860...")
cloudflare_cmd = ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"]
proc_cloudflare = subprocess.Popen(
    cloudflare_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Give cloudflared a few seconds to initialize logs
time.sleep(3)

# We'll store the discovered tunnel URL in `tunnel_url`
tunnel_url = None

def extract_cloudflare_url(log_line: str):
    # We look for "https://xxxxx.trycloudflare.com"
    match = re.search(r"https://.*?\.trycloudflare\.com\S*", log_line)
    return match.group(0) if match else None

print("[INFO] Searching cloudflared logs for the public URL...")

# Check up to 10 seconds for a log line containing the tunnel URL
start_time = time.time()
while (time.time() - start_time) < 10:
    if proc_cloudflare.poll() is not None:
        # cloudflared has exited or failed
        break

    line = proc_cloudflare.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue

    possible_url = extract_cloudflare_url(line)
    if possible_url:
        tunnel_url = possible_url
        break

if tunnel_url:
    print(f"[INFO] Found Cloudflare Tunnel URL => {tunnel_url}")
else:
    print("[WARNING] Could not parse the Cloudflare URL from logs yet.")

# Step 4) Launch either app.py or kaiser-app.py in the 'trellis' environment, unbuffered
app_path = os.path.join(TARGET_DIR, APP_FILE)
print(f"[INFO] Launching TRELLIS app ({APP_FILE}) in the 'trellis' environment...")
p_app = subprocess.Popen(
    ["conda", "run", "-n", "trellis", "python", "-u", app_path],
    cwd=TARGET_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Step 5) Stream logs from the selected app until it prints "Running on local URL:"
launch_detected = False
try:
    for line in iter(p_app.stdout.readline, ""):
        if not line:
            break

        print(line, end="")

        # If the app prints "Running on local URL:", we assume the server is up
        if "Running on local URL:" in line:
            launch_detected = True
            if tunnel_url:
                print(f"\n[INFO] {APP_FILE} is live! Access via Cloudflare:\n{tunnel_url}\n")
            else:
                print("\n[WARNING] We have not found a Cloudflare URL, but the app claims to be running.\n")

except KeyboardInterrupt:
    print("[INFO] Interrupted by user. Stopping the app.")
    p_app.kill()
finally:
    p_app.wait()

# If the app never printed "Running on local URL:", show the URL (if any)
if not launch_detected and tunnel_url:
    print("\n[INFO] The app didn't print 'Running on local URL:' but here's the Cloudflare URL anyway:")
    print(tunnel_url)

print("[INFO] Done. If you'd like to relaunch, just re-run this cell or the entire notebook.")


[INFO] Downloading kaiser-app.py from GitHub into /content/TRELLIS ...
[INFO] Installing/Updating cloudflared...
[INFO] Ensuring Gradio is installed in 'trellis' environment...
[INFO] Starting cloudflared (Quick Tunnel) on port 7860...
[INFO] Searching cloudflared logs for the public URL...
[INFO] Found Cloudflare Tunnel URL => https://nutten-committees-rangers-regulations.trycloudflare.com
[INFO] Launching TRELLIS app (kaiser-app.py) in the 'trellis' environment...
Traceback (most recent call last):
  File "/content/TRELLIS/kaiser-app.py", line 15, in <module>
    import gradio as gr
  File "/usr/local/envs/trellis/lib/python3.11/site-packages/gradio/__init__.py", line 3, in <module>
    import gradio._simple_templates
  File "/usr/local/envs/trellis/lib/python3.11/site-packages/gradio/_simple_templates/__init__.py", line 1, in <module>
    from .simpledropdown import SimpleDropdown
  File "/usr/local/envs/trellis/lib/python3.11/site-packages/gradio/_simple_templates/simpledropdown.py

In [36]:
from google.colab import files
from pathlib import Path
import os
import shutil
import subprocess

# STEP 1) Selecionar uma única imagem
uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Selecione exatamente uma imagem.")

image_name, image_bytes = next(iter(uploaded.items()))
image_path = f"/content/{image_name}"
Path(image_path).write_bytes(image_bytes)

# STEP 2) Confirmar TRELLIS e Conda
trellis_dir = "/content/TRELLIS"
if not os.path.isdir(trellis_dir):
    raise FileNotFoundError("Não encontrei /content/TRELLIS. Execute primeiro a célula de instalação.")

conda_bin = shutil.which("conda")
if not conda_bin:
    for candidate in ("/usr/local/bin/conda", "/usr/local/condabin/conda"):
        if os.path.isfile(candidate):
            conda_bin = candidate
            break

if not conda_bin:
    raise FileNotFoundError("Não encontrei o executável do Conda.")

# Sincronizar dependências e corrigir huggingface_hub
print("Verificando dependências e corrigindo Hugging Face no ambiente 'trellis'...")
subprocess.run([
    conda_bin, "run", "-n", "trellis",
    "pip", "install", "numpy==1.26.4", "huggingface_hub==0.25.2", "transformers==4.40.0", "gradio==4.44.1"
], check=True)

# STEP 3) Preparar o script de geração
output_path = "/content/apresentador_3d.glb"
log_file_path = "/content/log_geracao.txt"

script = r'''
import os
import gc
import sys
import torch

os.environ["SPCONV_ALGO"] = "native"
os.environ["ATTN_BACKEND"] = "sdpa"

torch.cuda.empty_cache()
gc.collect()

from PIL import Image
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils

if not torch.cuda.is_available():
    raise RuntimeError("A GPU CUDA não está disponível neste runtime.")

print("GPU detectada:", torch.cuda.get_device_name(0))
print("Carregando o modelo oficial TRELLIS-image-large...")

pipeline = TrellisImageTo3DPipeline.from_pretrained(
    "microsoft/TRELLIS-image-large"
)
pipeline.to("cuda")

if hasattr(pipeline, "enable_attention_slicing"):
    pipeline.enable_attention_slicing()

torch.cuda.empty_cache()
gc.collect()

image = Image.open(os.environ["TRELLIS_IMAGE"])
print("Gerando o modelo 3D (Configuração Ultra Light)...", flush=True)

try:
    with torch.inference_mode(), torch.cuda.amp.autocast(enabled=True, dtype=torch.float16):
        outputs = pipeline.run(
            image,
            seed=1,
            formats=["gaussian", "mesh"],
            preprocess_image=True,
            sparse_structure_sampler_params={
                "steps": 8,
                "cfg_strength": 5.5,
            }
        )

    print("Exportando arquivo GLB...", flush=True)
    glb = postprocessing_utils.to_glb(
        outputs["gaussian"][0],
        outputs["mesh"][0],
        simplify=0.90,
        texture_size=512,
    )
    glb.export(os.environ["TRELLIS_OUTPUT"])
    print("Arquivo salvo com sucesso em:", os.environ["TRELLIS_OUTPUT"], flush=True)
except Exception as e:
    import traceback
    print("\n--- [ERRO DETECTADO NO PYTHON INTERNO] ---", file=sys.stderr, flush=True)
    traceback.print_exc(file=sys.stderr)
    sys.exit(1)

finally:
    torch.cuda.empty_cache()
    gc.collect()
'''

# STEP 4) Executar no ambiente Conda 'trellis' salvando log físico
env = os.environ.copy()
env["TRELLIS_IMAGE"] = image_path
env["TRELLIS_OUTPUT"] = output_path
env["CUDA_HOME"] = "/usr/local/cuda-11.8"
env["PATH"] = "/usr/local/cuda-11.8/bin:" + env.get("PATH", "")
env["LD_LIBRARY_PATH"] = "/usr/local/cuda-11.8/lib64:/usr/lib64-nvidia:/usr/lib/x86_64-linux-gnu:" + env.get("LD_LIBRARY_PATH", "")

print(f"Iniciando o processo de geração. Logs em tempo real sendo gravados em: {log_file_path}")

try:
    with open(log_file_path, "w") as log_file:
        process = subprocess.Popen(
            [
                conda_bin, "run", "--no-capture-output",
                "-n", "trellis",
                "python", "-u", "-c", script,
            ],
            cwd=trellis_dir,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )

        for line in iter(process.stdout.readline, ""):
            print(line, end="")
            log_file.write(line)
            log_file.flush()

        process.wait()
        if process.returncode != 0:
            raise subprocess.CalledProcessError(process.returncode, process.args)

except Exception as e:
    print(f"\n[ERRO DE EXECUÇÃO]: {e}")
    print(f"Por favor, verifique o arquivo '{log_file_path}' no menu lateral esquerdo para ler o erro completo.")
    raise e

# STEP 5) Baixar o modelo 3D
if not os.path.isfile(output_path):
    raise FileNotFoundError("A geração terminou sem criar o arquivo GLB.")

files.download(output_path)


Saving apresentador-base-blender-recorte-alpha.png to apresentador-base-blender-recorte-alpha (25).png
Instalando dependências do ambiente 'trellis'...
Instalando xFormers para PyTorch 2.3 e CUDA 11.8...
Iniciando a geração. Log salvo também em: /content/log_geracao.txt
[SPARSE] Backend: spconv, Attention: xformers
Warp 1.18.0 initialized:
   CUDA Toolkit 13.4, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.18.0
Correções de dtype da malha ativadas.
GPU detectada: Tesla T4
Carregando o modelo TRELLIS-image-large...
[SPARSE][CONV] spconv algo: native
[ATTENTION] Using backend: xformers
Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_ma

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>